In [1]:
import json
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, fbeta_score, precision_recall_curve
from xgboost import XGBClassifier

print(f"Project root: {ROOT_DIR}")

Project root: c:\Users\MyBook Hype AMD\Documents\Projects\Chessys


In [2]:
PROCESSED_DIR = ROOT_DIR / "data" / "processed"
ARTIFACTS_DIR = ROOT_DIR / "artifacts"

with open(ARTIFACTS_DIR / "feature_names.json", "r") as f:
    feature_names = json.load(f)

train_df = pd.read_parquet(PROCESSED_DIR / "train.parquet")
val_df = pd.read_parquet(PROCESSED_DIR / "val.parquet")

X_train, y_train = train_df[feature_names], train_df["is_blunder"]
X_val, y_val = val_df[feature_names], val_df["is_blunder"]

num_neg = (y_train == 0).sum()
num_pos = (y_train == 1).sum()
scale_pos_weight = num_neg / num_pos

print(f"Dimensi X_train: {X_train.shape} | Proporsi Blunder: {y_train.mean()*100:.2f}%")
print(f"Dimensi X_val  : {X_val.shape} | Proporsi Blunder: {y_val.mean()*100:.2f}%")
print(f"Dihitung scale_pos_weight: {scale_pos_weight:.2f}")

Dimensi X_train: (7000, 12) | Proporsi Blunder: 9.93%
Dimensi X_val  : (1500, 12) | Proporsi Blunder: 9.93%
Dihitung scale_pos_weight: 9.07


In [3]:
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    early_stopping_rounds=25,
    random_state=42,
    n_jobs=-1,
)

xgb_model.fit(
    X_train, y_train, eval_set=[(X_train, y_train), (X_val, y_val)], verbose=50
)

print(f"\nTraining berhenti pada iterasi terbaik: {xgb_model.best_iteration}")
print(f"Validation Log-Loss terbaik: {xgb_model.best_score:.4f}")

[0]	validation_0-logloss:0.68710	validation_1-logloss:0.68778
[50]	validation_0-logloss:0.56370	validation_1-logloss:0.59002
[100]	validation_0-logloss:0.51448	validation_1-logloss:0.56406
[150]	validation_0-logloss:0.47480	validation_1-logloss:0.54556
[200]	validation_0-logloss:0.43850	validation_1-logloss:0.52603
[250]	validation_0-logloss:0.40570	validation_1-logloss:0.50922
[299]	validation_0-logloss:0.37744	validation_1-logloss:0.49467

Training berhenti pada iterasi terbaik: 298
Validation Log-Loss terbaik: 0.4944


In [4]:
val_probs = xgb_model.predict_proba(X_val)[:, 1]

thresholds = np.linspace(0.1, 0.9, 81)
best_threshold = 0.5
best_f2 = 0.0

threshold_logs = []

for th in thresholds:
    preds = (val_probs >= th).astype(int)
    f2 = fbeta_score(y_val, preds, beta=2, zero_division=0)
    threshold_logs.append({"threshold": th, "f2_score": f2})

    if f2 > best_f2:
        best_f2 = f2
        best_threshold = th

threshold_df = pd.DataFrame(threshold_logs)
print(f"Optimal Threshold (F2-Score): {best_threshold:.2f} (F2 Score = {best_f2:.4f})")

default_preds = (val_probs >= 0.5).astype(int)
optimal_preds = (val_probs >= best_threshold).astype(int)

print("\n--- Evaluasi Threshold Default (0.50) pada Validasi ---")
print(classification_report(y_val, default_preds, digits=4))

print(f"--- Evaluasi Optimal Threshold ({best_threshold:.2f}) pada Validasi ---")
print(classification_report(y_val, optimal_preds, digits=4))

Optimal Threshold (F2-Score): 0.22 (F2 Score = 0.4078)

--- Evaluasi Threshold Default (0.50) pada Validasi ---
              precision    recall  f1-score   support

           0     0.9186    0.7602    0.8319      1351
           1     0.1518    0.3893    0.2185       149

    accuracy                         0.7233      1500
   macro avg     0.5352    0.5747    0.5252      1500
weighted avg     0.8424    0.7233    0.7710      1500

--- Evaluasi Optimal Threshold (0.22) pada Validasi ---
              precision    recall  f1-score   support

           0     0.9558    0.4160    0.5797      1351
           1     0.1349    0.8255    0.2319       149

    accuracy                         0.4567      1500
   macro avg     0.5453    0.6207    0.4058      1500
weighted avg     0.8742    0.4567    0.5451      1500



In [5]:
raw_model_path = ARTIFACTS_DIR / "blunder_xgb_raw.joblib"
joblib.dump(xgb_model, raw_model_path)

threshold_metadata = {
    "optimal_threshold": float(best_threshold),
    "best_f2_score": float(best_f2),
    "scale_pos_weight": float(scale_pos_weight),
}

with open(ARTIFACTS_DIR / "model_config.json", "w") as f:
    json.dump(threshold_metadata, f, indent=2)

print(f"Base model tersimpan di: {raw_model_path}")
print(f"Metadata ambang batas tersimpan di: {ARTIFACTS_DIR / 'model_config.json'}")

✓ Base model tersimpan di: c:\Users\MyBook Hype AMD\Documents\Projects\Chessys\artifacts\blunder_xgb_raw.joblib
✓ Metadata ambang batas tersimpan di: c:\Users\MyBook Hype AMD\Documents\Projects\Chessys\artifacts\model_config.json
